In [5]:
import numpy as np
import pandas as pd
import os
import glob
import re

# 1. Table Prep
Prepare the table: add names, make the format readable, etc.

In [2]:
# read in the distinctiveness ratings
dist_rat_df = pd.read_csv("Average_rock_distinct_ratings.txt", 
                          sep=r"\s+", header=None, 
                          names=['image_id', 'Category','Item', 'Distinct_Rating'])
dist_rat_df.head()

,image_id,Category,Item,Distinct_Rating
0,1,1,1,3.99
1,2,1,2,3.93
2,3,1,3,3.17
3,4,1,4,3.18
4,5,1,5,3.32


In [20]:
dist_rat_df = dist_rat_df[['image_id', 'Distinct_Rating']]
dist_rat_df.rename(columns={'Distinct_Rating':'distinct_rat'}, inplace=True)
dist_rat_df

,image_id,distinct_rat
0,1,3.99
1,2,3.93
2,3,3.17
3,4,3.18
4,5,3.32
...,...,...
475,476,2.48
476,477,1.88
477,478,2.68
478,479,1.84


In [21]:
dist_rat_df.to_csv('Rocks480_distinctiveness.csv', index=False)

In [6]:
# --- Config ---
IMG_DIR = "Stimuli_Rocks480"
OUTPUT_CSV = "images.csv"

# --- Parse filenames ---
records = []

for filename in sorted(os.listdir(IMG_DIR)):
    # Match pattern: Category_Subcategory_Item.ext
    match = re.match(r'^([^_]+)_([^_]+)_(\d+)\.(png|jpg|jpeg|webp)$', filename, re.IGNORECASE)
    if not match:
        print(f"Skipping (unrecognized format): {filename}")
        continue

    category, subcategory, item, ext = match.groups()

    records.append({
        "filename":    filename,
        "category":    category,
        "sub_category": subcategory,
        "sub_cat_item": int(item), 
    })

# --- Build DataFrame ---
df = pd.DataFrame(records, columns=["filename", "category", "sub_category", "sub_cat_item"])
df['image_id'] = range(1,481)

In [12]:
df_merged = df.merge(dist_rat_df[['image_id', 'Distinct_Rating']], on='image_id')
df_merged.rename(columns={'Distinct_Rating':'distinct_rat'}, inplace=True)

In [18]:
df = df[['image_id','category','sub_category','sub_cat_item','filename']]

In [19]:
df.to_csv('Rocks480.csv',index=False)

In [13]:
df_merged.columns

Index(['filename', 'category', 'sub_category', 'sub_cat_item', 'image_id',
       'distinct_rat'],
      dtype='str')

In [15]:
df_merged = df_merged[['image_id','category','sub_category','sub_cat_item','distinct_rat','filename']]

In [7]:
df

,filename,category,sub_category,sub_cat_item,image_id
0,I_Andesite_01.png,I,Andesite,1,1
1,I_Andesite_02.png,I,Andesite,2,2
2,I_Andesite_03.png,I,Andesite,3,3
3,I_Andesite_04.png,I,Andesite,4,4
4,I_Andesite_05.png,I,Andesite,5,5
...,...,...,...,...,...
475,S_Shale_12.png,S,Shale,12,476
476,S_Shale_13.png,S,Shale,13,477
477,S_Shale_14.png,S,Shale,14,478
478,S_Shale_15.png,S,Shale,15,479


In [7]:
# Define folder path
folder = "Stimuli_Rocks480"

# Get all .png files, sorted
filenames = sorted(glob.glob(os.path.join(folder, '*.png')))

# Add filenames to the dataframe
dist_rat_df["image_png"] = [os.path.basename(f) for f in filenames]

# Extract category info from filenames
dist_rat_df["Broad_Category_Name"] = dist_rat_df["image_png"].str.split("_").str[0]
dist_rat_df["Category_Name"] = dist_rat_df["image_png"].str.split("_").str[1]

dist_rat_df.head()

,image_id,Category,Item,Distinct_Rating,image_png,Broad_Category_Name,Category_Name
0,1,1,1,3.99,I_Andesite_01.png,I,Andesite
1,2,1,2,3.93,I_Andesite_02.png,I,Andesite
2,3,1,3,3.17,I_Andesite_03.png,I,Andesite
3,4,1,4,3.18,I_Andesite_04.png,I,Andesite
4,5,1,5,3.32,I_Andesite_05.png,I,Andesite


# 2. Group Creation
Create two groups based on their distinctiveness ratings

Assign like:

- Target
- Foil
- Foil
- Target
-----------
- Target
- Foil
- Foil
- Target

In [8]:
# Sort by Category first to keep order stable
dist_rat_df = dist_rat_df.sort_values(["Category", "Distinct_Rating"], ascending=[True, False]).copy()

# We'll store the results here
dist_rat_df["Group1"] = ""
dist_rat_df["Group2"] = ""

In [10]:
# Process each category separately
start_with_target = True  # Category 1 starts with Target in Group1
for cat, group_df in dist_rat_df.groupby("Category", sort=False):
    # Determine the pattern for this category
    if start_with_target:
        patterns = [("Target", "Foil"), ("Foil", "Target")]
    else:
        patterns = [("Foil", "Target"), ("Target", "Foil")]
    
    # Assign patterns row by row
    for i, idx in enumerate(group_df.index):
        g1, g2 = patterns[i % 2]  # alternate
        dist_rat_df.at[idx, "Group1"] = g1
        dist_rat_df.at[idx, "Group2"] = g2
    
    # Flip pattern for next category
    start_with_target = not start_with_target

# Reset index
dist_rat_df = dist_rat_df.reset_index(drop=True)

In [12]:
# Make sure no objects are both targets and foils
(dist_rat_df["Group1"] == dist_rat_df["Group2"]).sum()

0

In [13]:
# Reorder columns
dist_rat_df = dist_rat_df[['image_id', 'Broad_Category_Name', 'Category', 'Category_Name', 'Item', 'Distinct_Rating', 'Group1','Group2','image_png']]
dist_rat_df.head()

,image_id,Broad_Category_Name,Category,Category_Name,Item,Distinct_Rating,Group1,Group2,image_png
0,7,I,1,Andesite,7,4.81,Target,Foil,I_Andesite_07.png
1,14,I,1,Andesite,14,4.47,Foil,Target,I_Andesite_14.png
2,1,I,1,Andesite,1,3.99,Target,Foil,I_Andesite_01.png
3,15,I,1,Andesite,15,3.98,Foil,Target,I_Andesite_15.png
4,2,I,1,Andesite,2,3.93,Target,Foil,I_Andesite_02.png


In [17]:
positions = [0, 1, 4, 5, 8, 9, 12, 13]
dist_rat_df.groupby("Category", group_keys=False).nth(positions)

,image_id,Broad_Category_Name,Category,Category_Name,Item,Distinct_Rating,Group1,Group2,image_png
0,7,I,1,Andesite,7,4.81,Target,Foil,I_Andesite_07.png
1,14,I,1,Andesite,14,4.47,Foil,Target,I_Andesite_14.png
4,2,I,1,Andesite,2,3.93,Target,Foil,I_Andesite_02.png
5,12,I,1,Andesite,12,3.75,Foil,Target,I_Andesite_12.png
8,11,I,1,Andesite,11,3.35,Target,Foil,I_Andesite_11.png
...,...,...,...,...,...,...,...,...,...
469,465,S,30,Shale,1,2.70,Target,Foil,S_Shale_01.png
472,472,S,30,Shale,8,2.55,Foil,Target,S_Shale_08.png
473,476,S,30,Shale,12,2.48,Target,Foil,S_Shale_12.png
476,467,S,30,Shale,3,1.95,Foil,Target,S_Shale_03.png


In [18]:
# choose only half of the data
# get only 0th, 1st, 4th, 5th, 8th... elements from each category
# to get similar distinctiveness ratings in different categories
positions = [0, 1, 4, 5, 8, 9, 12, 13]

half_df = (dist_rat_df
           .groupby("Category", group_keys=False).nth(positions)
           .reset_index(drop=True)
           .copy()
           )
half_df

,image_id,Broad_Category_Name,Category,Category_Name,Item,Distinct_Rating,Group1,Group2,image_png
0,7,I,1,Andesite,7,4.81,Target,Foil,I_Andesite_07.png
1,14,I,1,Andesite,14,4.47,Foil,Target,I_Andesite_14.png
2,2,I,1,Andesite,2,3.93,Target,Foil,I_Andesite_02.png
3,12,I,1,Andesite,12,3.75,Foil,Target,I_Andesite_12.png
4,11,I,1,Andesite,11,3.35,Target,Foil,I_Andesite_11.png
...,...,...,...,...,...,...,...,...,...
235,465,S,30,Shale,1,2.70,Target,Foil,S_Shale_01.png
236,472,S,30,Shale,8,2.55,Foil,Target,S_Shale_08.png
237,476,S,30,Shale,12,2.48,Target,Foil,S_Shale_12.png
238,467,S,30,Shale,3,1.95,Foil,Target,S_Shale_03.png


Columns:

1. `image_id`: Order number in the original list (and in the folder)
2. `Broad_Category_Name`: I - igneous, M - metamorphic, S - sedimentary rocks
3. `Category`: Category ID
4. `Category_Name`
5. `Item`: There are 16 items of each category
6. `Distinct_Rating`: Average distinctiveness ratings from Meagher & Nosofsky (2023)
7. `Group1`: Whether this object is a target (old) or a foil (new) for the first group of subjects (opposite to `Group2`)
8. `Group2`: Whether this object is a target (old) or a foil (new) for the second group of subjects (opposite to `Group1`)
9. `image_png`: Image file name with this object

In [12]:
# Save data
# half_df.to_csv("Average_Distinct_Ratings_480Rocks.csv", index=False)
# half_df.to_json("Average_Distinct_Ratings_480Rocks.json", orient="records", indent=2)

dist_rat_df.to_json("Average_Distinct_Ratings_480Rocks.json", orient="records", indent=2)

In [26]:
half_df

,image_id,Broad_Category_Name,Category,Category_Name,Item,Distinct_Rating,Group1,Group2,image_png
0,7,I,1,Andesite,7,4.81,Target,Foil,I_Andesite_07.png
1,14,I,1,Andesite,14,4.47,Foil,Target,I_Andesite_14.png
2,2,I,1,Andesite,2,3.93,Target,Foil,I_Andesite_02.png
3,12,I,1,Andesite,12,3.75,Foil,Target,I_Andesite_12.png
4,11,I,1,Andesite,11,3.35,Target,Foil,I_Andesite_11.png
...,...,...,...,...,...,...,...,...,...
235,465,S,30,Shale,1,2.70,Target,Foil,S_Shale_01.png
236,472,S,30,Shale,8,2.55,Foil,Target,S_Shale_08.png
237,476,S,30,Shale,12,2.48,Target,Foil,S_Shale_12.png
238,467,S,30,Shale,3,1.95,Foil,Target,S_Shale_03.png


# Make 8 MDS table nicer

In [28]:
mds8 = pd.read_csv(
    "480Rocks_8MDS_Solution.dat",
    sep=r"\s+",
    header=None,
    names=["dim" + str(i) for i in range(1, 9)]
)
mds8['image_id'] = np.arange(1, len(mds8) + 1)

mds8

,dim1,dim2,dim3,dim4,dim5,dim6,dim7,dim8,image_id
0,-2.757,-1.594,1.052,0.768,1.343,1.711,1.795,0.809,1
1,2.001,-0.754,1.405,-1.382,-1.384,0.976,-0.067,1.972,2
2,0.261,-0.508,-2.962,-0.608,0.009,-0.191,2.720,-1.224,3
3,0.382,-0.027,-0.148,-1.899,-1.948,1.862,0.968,0.986,4
4,0.288,2.013,-1.102,-1.285,0.344,-0.009,2.696,-0.174,5
...,...,...,...,...,...,...,...,...,...
475,0.749,-3.021,-1.908,-1.689,-0.493,4.441,-0.122,1.618,476
476,-0.054,-2.743,-1.435,-2.903,0.315,3.161,1.833,1.288,477
477,0.007,-2.077,-4.012,-1.633,-0.366,2.558,4.210,-0.060,478
478,0.177,-2.520,-2.293,-1.216,0.545,0.805,2.302,0.629,479


In [29]:
# save mds8
mds8.to_csv("MDS8_solution.csv", index=False)